# CIFAR-10 – täysin yhdistetty neuroverkko (FCN)

## Johdanto

Tässä notebookissa luokitellaan CIFAR-10-kuvia täysin yhdistetyllä neuroverkolla (FCN). FCN:ssä kuva litistetään yhdeksi 3 072 luvun vektoriksi (32 × 32 × 3), ja jokainen neuroni on yhteydessä jokaiseen syötteen pikseliin. Pohjana on käytetty MNIST-datasetille tehtyä Keras-työkirjaa, jota on muokattu värikuville sopivaksi.

Toteutetaan kaksi mallia: yksinkertainen perusmalli ja leveämpi malli, jossa on Batch Normalization ja Dropout. Tulokset tallennetaan tiedostoon `results_fcn.json`, jota CNN-notebook käyttää FCN:n ja CNN:n vertailussa. **Tämä notebook ajetaan ennen CNN-notebookia.**

Tehtävänannon tavoitteet FCN:lle: realistinen 50–55 %, kunnianhimoinen 57–60 % testitarkkuus.

In [ ]:
EPOCHS = 60
PATIENCE = 8

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

import time, json
import numpy as np
import matplotlib.pyplot as plt
import keras
from keras import layers
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

keras.utils.set_random_seed(42)
print("Keras", keras.__version__, "| backend:", keras.backend.backend())

## Data

CIFAR-10 sisältää 60 000 värikuvaa (32×32 pikseliä, 3 värikanavaa) kymmenestä luokasta: lentokone, auto, lintu, kissa, peura, koira, sammakko, hevonen, laiva ja kuorma-auto. Opetusdataa on 50 000 ja testidataa 10 000 kuvaa, ja luokat ovat tasakokoisia (5 000 / 1 000 kuvaa luokkaa kohden), joten tarkkuus on järkevä päämittari.

Verrattuna MNISTiin data on selvästi vaikeampaa: kohde voi olla missä kohtaa kuvaa tahansa, eri kokoisena ja eri kulmasta, taustat vaihtelevat, ja osa luokista muistuttaa toisiaan (kissa/koira, auto/kuorma-auto, peura/hevonen). Pieni resoluutio vaikeuttaa tunnistamista myös ihmiselle. Alla on 20 ensimmäistä opetuskuvaa nimikkeineen.

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
y_train, y_test = y_train.flatten(), y_test.flatten()

class_names = ["airplane", "automobile", "bird", "cat", "deer",
               "dog", "frog", "horse", "ship", "truck"]

print("train:", x_train.shape, "test:", x_test.shape)

In [ ]:
plt.figure(figsize=(10, 4))
for i in range(20):
    plt.subplot(2, 10, i + 1)
    plt.imshow(x_train[i])
    plt.title(class_names[y_train[i]], fontsize=7)
    plt.axis("off")
plt.tight_layout()
plt.show()

## Esikäsittely

Pikseliarvot muutetaan ensin liukuluvuiksi ja skaalataan välille 0–1. Sen jälkeen jokainen värikanava (R, G, B) standardoidaan erikseen: kanavasta vähennetään sen keskiarvo ja jaetaan keskihajonnalla. Kanavat käsitellään erikseen, koska niiden jakaumat poikkeavat toisistaan (esim. sininen kanava on keskimäärin eri tasolla kuin punainen). Standardoitu syöte on keskitetty nollan ympärille, mikä tasoittaa gradientteja ja nopeuttaa ja vakauttaa oppimista.

Keskiarvo ja hajonta lasketaan vain opetusdatasta ja samoja arvoja käytetään testidatalle. Näin testidatasta ei vuoda tietoa malliin, ja testitulos kuvaa aidosti suoriutumista ennennäkemättömällä datalla. Värikuvan muoto (32, 32, 3) säilyy esikäsittelyssä ennallaan.

In [ ]:
# Skaalaus 0-1 ja kanavakohtainen standardointi (keskiarvo ja hajonta lasketaan vain opetusdatasta)
x_train_f = x_train.astype("float32") / 255.0
x_test_f = x_test.astype("float32") / 255.0

mean = x_train_f.mean(axis=(0, 1, 2))
std = x_train_f.std(axis=(0, 1, 2))
print("kanavien keskiarvot (R,G,B):", mean.round(3), "hajonnat:", std.round(3))

x_train_n = (x_train_f - mean) / std
x_test_n = (x_test_f - mean) / std

## Mallit

**Perusmalli (`fcn_baseline`)**: Flatten → Dense 512 → Dense 256 → Dense 10 (softmax). Rakenne vastaa MNIST-mallia, mutta syötteessä on kolminkertainen määrä arvoja värikanavien takia.

**Parannettu malli (`fcn_improved`)**: kolme leveämpää piilokerrosta (1024 → 512 → 256), joissa jokaisessa on Dense → BatchNormalization → ReLU → Dropout.
- *Batch Normalization* normalisoi kerroksen ulostulot erä kerrallaan. Se vakauttaa ja nopeuttaa oppimista ja toimii lievänä regularisoijana. Koska BatchNorm sisältää oman siirtotermin, Dense-kerroksen bias on turha (`use_bias=False`).
- *Dropout* (0.3–0.4) nollaa koulutuksen aikana satunnaisen osan neuroneista, jolloin verkko ei voi nojata yksittäisiin neuroneihin ja ylisovittuminen vähenee.

Leveämpi verkko pystyy oppimaan enemmän, mutta ilman regularisointia se ylisovittuisi entistä nopeammin, joten leveys ja regularisointi kulkevat tässä yhdessä. Huomattavaa on parametrien suuri määrä: jo ensimmäinen kerros sisältää 3 072 × 1 024 ≈ 3,1 miljoonaa painoa.

In [ ]:
def fcn_baseline():
    return keras.Sequential([
        keras.Input(shape=(32, 32, 3)),
        layers.Flatten(),
        layers.Dense(512, activation="relu"),
        layers.Dense(256, activation="relu"),
        layers.Dense(10, activation="softmax"),
    ], name="fcn_baseline")


def fcn_block(units, dropout):
    return [layers.Dense(units, use_bias=False),
            layers.BatchNormalization(),
            layers.Activation("relu"),
            layers.Dropout(dropout)]


def fcn_improved():
    return keras.Sequential([
        keras.Input(shape=(32, 32, 3)),
        layers.Flatten(),
        *fcn_block(1024, 0.4),
        *fcn_block(512, 0.4),
        *fcn_block(256, 0.3),
        layers.Dense(10, activation="softmax"),
    ], name="fcn_improved")


fcn_improved().summary()

## Koulutus

Kaikki mallit koulutetaan samalla `train`-funktiolla, jotta vertailu on reilu: Adam-optimoija (oppimisnopeus 0.001), häviöfunktiona `sparse_categorical_crossentropy` (luokat ovat kokonaislukuja 0–9), eräkoko 128 ja 10 % opetusdatasta validaatioon.

Kaksi callbackia ohjaa koulutusta:
- **EarlyStopping** lopettaa koulutuksen, kun validaatiohäviö ei ole parantunut `PATIENCE` epookkiin, ja palauttaa parhaan epookin painot. Näin epookkien enimmäismäärä (60) toimii vain ylärajana, eikä ylisovittunutta mallia päädytä käyttämään.
- **ReduceLROnPlateau** puolittaa oppimisnopeuden, jos validaatiohäviö jumittaa kolme epookkia. Pienempi askel auttaa hienosäätämään painoja koulutuksen loppuvaiheessa.

Opetusaika mitataan `time.perf_counter()`-funktiolla `fit`-kutsun ympäriltä. Aika riippuu laitteistosta (CPU/GPU), joten se on vertailukelpoinen vain saman koneen ajojen kesken.

In [ ]:
def train(model, x, y, epochs, batch_size=128, lr=1e-3):
    model.compile(optimizer=keras.optimizers.Adam(lr),
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE,
                                      restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3),
    ]
    start = time.perf_counter()
    history = model.fit(x, y, validation_split=0.1, epochs=epochs,
                        batch_size=batch_size, callbacks=callbacks, verbose=2)
    duration = time.perf_counter() - start
    return history, duration

In [ ]:
baseline = fcn_baseline()
h_base, t_base = train(baseline, x_train_n, y_train, EPOCHS)

In [ ]:
improved = fcn_improved()
h_imp, t_imp = train(improved, x_train_n, y_train, EPOCHS)

## Oppimiskäyrät

Kuvaajissa yhtenäinen viiva on opetusdata ja katkoviiva validaatiodata. Olennaista on näiden välinen ero: jos opetushäviö laskee mutta validaatiohäviö kääntyy nousuun, malli ylisovittuu eli opettelee opetuskuvien yksityiskohtia yleisten piirteiden sijaan.

Perusmallissa ilman regularisointia ero kasvaa nopeasti, ja validaatiohäviö alkaa nousta jo muutaman epookin jälkeen, minkä vuoksi early stopping pysäyttää sen aikaisin. Regularisoidussa mallissa käyrät pysyvät lähempänä toisiaan ja validaatiotarkkuus jatkaa nousua pidempään. Kohdissa, joissa oppimisnopeus puolittuu, käyrissä näkyy usein pieni porras alaspäin (häviö) tai ylöspäin (tarkkuus).

In [ ]:
HISTORIES = {'perus': h_base, 'parannettu': h_imp}
def plot_history(histories):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    for name, h in histories.items():
        ax[0].plot(h.history["loss"], label=f"{name} train")
        ax[0].plot(h.history["val_loss"], "--", label=f"{name} val")
        ax[1].plot(h.history["accuracy"], label=f"{name} train")
        ax[1].plot(h.history["val_accuracy"], "--", label=f"{name} val")
    ax[0].set_title("Loss"); ax[1].set_title("Accuracy")
    for a in ax:
        a.set_xlabel("epookki"); a.legend(fontsize=8); a.grid(alpha=0.3)
    plt.show()

plot_history(HISTORIES)

## Testitulokset

Taulukko näyttää testitarkkuuden, parametrien määrän, opetusajan ja epookkien määrän molemmille malleille. Testidataa käytetään vain tässä lopullisessa arvioinnissa; mallien valinta ja early stopping perustuvat validaatiodataan.

Parannettu malli on parempi, vaikka siinä on moninkertainen määrä parametreja: regularisointi pitää ylisovittumisen kurissa ja sallii pidemmän koulutuksen. Alla olevassa solussa tuloksia verrataan tehtävänannon tavoitteisiin. Sen jälkeen parhaan mallin tuloksia tarkastellaan luokittain (classification report) ja sekaannusmatriisin avulla. Sekaannusmatriisista näkyy, että ajoneuvot ja laiva tunnistetaan parhaiten, kun taas eläinluokat – erityisesti kissa, lintu ja koira – ovat vaikeimpia ja sekoittuvat keskenään.

In [ ]:
X_TEST = x_test_n
MODELS = {'fcn_baseline': (baseline, h_base, t_base),
          'fcn_improved': (improved, h_imp, t_imp)}
results = {}
for name, (m, h, t) in MODELS.items():
    loss, acc = m.evaluate(X_TEST, y_test, verbose=0)
    results[name] = {"test_acc": float(acc), "params": int(m.count_params()),
                     "train_time_s": round(t, 1), "epochs": len(h.history["loss"])}
    print(f"{name:12s} tarkkuus {acc:.4f} | parametrit {m.count_params():,} | "
          f"aika {t:.0f} s | epookit {len(h.history['loss'])}")

In [ ]:
# verrataan tuloksia tehtävänannon tavoitteisiin
for name, r in results.items():
    acc = r["test_acc"] * 100
    taso = ("kunnianhimoinen tavoite (57-60 %) saavutettu" if acc >= 57 else
            "realistinen tavoite (50-55 %) saavutettu" if acc >= 50 else "alle tavoitteen")
    print(f"{name:14s} {acc:5.1f} %  -> {taso}")

In [ ]:
# tallennetaan tulokset CNN-notebookin vertailua varten
results["fcn_improved"]["history"] = {k: [float(v) for v in vals]
                                      for k, vals in h_imp.history.items()
                                      if k in ("loss", "val_loss", "accuracy", "val_accuracy")}
with open("results_fcn.json", "w") as f:
    json.dump(results, f, indent=2)
best_model = improved

In [ ]:
probs = best_model.predict(X_TEST, verbose=0)
y_pred = probs.argmax(axis=1)

print(classification_report(y_test, y_pred, target_names=class_names, digits=3))

fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred, normalize="true"),
                       display_labels=class_names).plot(
    ax=ax, xticks_rotation=45, values_format=".2f", cmap="Blues", colorbar=False)
ax.set_title("Confusion matrix (rivit normalisoitu)")
plt.show()

## Todennäköisyysjakaumat

Softmax-kerros antaa jokaiselle kuvalle todennäköisyysjakauman kymmenen luokan yli. Palkkikuvioissa vihreä on oikea luokka ja punainen mallin ennuste, jos se on väärä.

Oikein luokitelluissa kuvissa todennäköisyys on usein keskittynyt lähes kokonaan yhteen luokkaan. Histogrammi näyttää mallin varmuuden (suurimman todennäköisyyden) jakauman oikein ja väärin luokitelluille kuville erikseen: oikeat ennusteet painottuvat lähelle 100 %:a, kun taas väärien ennusteiden varmuus on keskimäärin selvästi pienempi. Malli siis jossain määrin ”tietää” milloin se on epävarma, mikä on hyödyllistä: matalan varmuuden ennusteet voisi käytännön sovelluksessa ohjata tarkistettavaksi. Kaikkiin ennusteisiin ei silti voi luottaa, sillä osa virheistä tehdään suurella varmuudella.

In [ ]:
def show_with_probs(indices, title):
    fig, ax = plt.subplots(len(indices), 2, figsize=(8, 2.2 * len(indices)),
                           gridspec_kw={"width_ratios": [1, 3]})
    for row, i in enumerate(indices):
        ax[row, 0].imshow(x_test[i]); ax[row, 0].axis("off")
        ax[row, 0].set_title(f"oikea: {class_names[y_test[i]]}", fontsize=8)
        colors = ["tab:green" if c == y_test[i] else
                  ("tab:red" if c == y_pred[i] else "tab:gray") for c in range(10)]
        ax[row, 1].barh(class_names, probs[i], color=colors)
        ax[row, 1].set_xlim(0, 1); ax[row, 1].invert_yaxis()
        ax[row, 1].tick_params(labelsize=7)
    fig.suptitle(title)
    plt.tight_layout(); plt.show()

correct = np.where(y_pred == y_test)[0]
show_with_probs(correct[:4], "Oikein luokitellut (vihreä = oikea luokka)")

In [ ]:
conf = probs.max(axis=1)
plt.figure(figsize=(7, 3.5))
plt.hist(conf[y_pred == y_test], bins=30, alpha=0.7, label="oikein")
plt.hist(conf[y_pred != y_test], bins=30, alpha=0.7, label="väärin")
plt.xlabel("mallin varmuus (suurin todennäköisyys)"); plt.ylabel("kuvia")
plt.legend(); plt.show()
print(f"keskimääräinen varmuus: oikein {conf[y_pred == y_test].mean():.2f}, "
      f"väärin {conf[y_pred != y_test].mean():.2f}")

## Väärin luokitellut kuvat

Virheet on järjestetty mallin varmuuden mukaan, joten ensimmäisinä näkyvät tapaukset, joissa malli on ollut varmimmin väärässä. Tyypillisesti virheet osuvat samankaltaisten luokkien välille: kissa ↔ koira, auto ↔ kuorma-auto, peura ↔ hevonen ja lintu ↔ lentokone. Myös sekaannusmatriisin suurimmat diagonaalin ulkopuoliset arvot sijoittuvat näihin pareihin.

Osa virheistä on ymmärrettäviä myös ihmisen silmin: kohde on hyvin pieni, osittain rajattu pois kuvasta, epätavallisesta kulmasta kuvattu tai taustaltaan harhaanjohtava (esim. lintu sinistä taivasta vasten näyttää lentokoneelta, ja vesistö taustalla vetää kohti laiva-luokkaa). Muutama testikuva on myös tulkinnanvarainen. Tällaisia virheitä on vaikea korjata pelkällä arkkitehtuurilla 32×32-resoluutiolla.

In [ ]:
wrong = np.where(y_pred != y_test)[0]
# varmimmat virheet ensin
wrong = wrong[np.argsort(-conf[wrong])]

plt.figure(figsize=(12, 6))
for k, i in enumerate(wrong[:15]):
    plt.subplot(3, 5, k + 1)
    plt.imshow(x_test[i]); plt.axis("off")
    plt.title(f"oikea: {class_names[y_test[i]]}\nennuste: {class_names[y_pred[i]]} "
              f"({conf[i]:.0%})", fontsize=8)
plt.suptitle("Varmimmat väärät ennusteet")
plt.tight_layout(); plt.show()

show_with_probs(wrong[:4], "Väärin luokitellut (punainen = ennuste, vihreä = oikea)")

## Kokeilut ja johtopäätökset

**Kokeillut asiat**
- *Esikäsittely:* pelkän 0–1-skaalauksen sijaan käytetään kanavakohtaista standardointia, jolloin syöte on keskitetty nollaan ja jokaisen värikanavan vaihtelu on samaa luokkaa.
- *Arkkitehtuuri:* kapeasta kahden piilokerroksen verkosta (512–256) siirryttiin leveämpään kolmen kerroksen verkkoon (1024–512–256).
- *Regularisointi:* Batch Normalization ja Dropout jokaisessa piilokerroksessa.
- *Koulutus:* early stopping parhaiden painojen palautuksella ja oppimisnopeuden puolitus validaatiohäviön jumittuessa.

**Johtopäätökset**
- Perusmalli ylisovittuu nopeasti: opetustarkkuus jatkaa nousuaan, mutta validaatiohäviö kääntyy nousuun jo alkuvaiheessa.
- Regularisointi on FCN:lle välttämätöntä. Parannettu malli pystyy oppimaan pidempään ja saavuttaa paremman testitarkkuuden, vaikka parametreja on huomattavasti enemmän.
- FCN:n suorituskyvyllä on kuitenkin selvä katto. Koska kuva litistetään vektoriksi, verkko ei tiedä, mitkä pikselit ovat vierekkäin, ja saman kohteen siirtäminen kuvassa muuttaa syötteen kokonaan. Malli joutuu oppimaan jokaisen piirteen erikseen jokaiseen kuvan kohtaan, mikä vaatii valtavasti parametreja ja johtaa helposti ylisovittumiseen.
- Virheet keskittyvät eläinluokkiin, joissa erot ovat muodoissa ja yksityiskohdissa, eivät pelkissä väreissä tai yleisessä sävyssä. FCN näyttää nojaavan paljolti värijakaumiin ja taustaan (esim. sininen tausta → laiva tai lentokone).

Seuraava luonteva askel on konvoluutioverkko, joka hyödyntää kuvan paikallista rakennetta – sitä käsitellään CNN-notebookissa.